In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [ ]:
df = pd.read_csv(data)

In [ ]:
df.columns

In [ ]:
sns.histplot(df['fuel_efficiency_mpg'], bins=50)

In [ ]:
features = ['engine_displacement','horsepower','vehicle_weight','model_year']
target = 'fuel_efficiency_mpg'

In [ ]:
df = df[features + [target]]
X = df[features]
y = df[target]

There's one column with missing values. What is it?

In [ ]:
df.isnull().sum()

What's the median (50% percentile) for variable 'horsepower'?

In [ ]:
df['horsepower'].median()

In [ ]:
def split_dataset(df, seed):
    """Split dataset 60%/20%/20%."""
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)
    
    return df_train, df_val, df_test

In [ ]:
def train_linear_regression_from_lesson(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [ ]:
def rmse_from_lesson(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [ ]:
def cp_df_after_filling_hp(df_orig, fill_value):
    df = df_orig.copy()
    df["horsepower"] = df["horsepower"].fillna(fill_value)
    return df

In [ ]:
def train_and_calculate_error_from_lesson(df_train, df_val, fill_value):
    df = cp_df_after_filling_hp(df_train, fill_value)
    X_train = df[features].to_numpy()
    y_train = df[target].to_numpy()
    
    w0, w = train_linear_regression_from_lesson(X_train, y_train)
    
    df = cp_df_after_filling_hp(df_val, fill_value)
    X_val = df[features].to_numpy()
    y_val = df[target].to_numpy()
    
    y_pred = w0 + X_val.dot(w)
    score = rmse_from_lesson(y_val, y_pred)
    return round(score, 3)

- We need to deal with missing values for the column from Q1.
- We have two options: fill it with 0 or with the mean of this variable.
- Try both options. For each, train a linear regression model without regularization using the code from the lessons.
- For computing the mean, use the training only!
- Use the validation dataset to evaluate the models and compare the RMSE of each option.
- Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.
- Which option gives better RMSE?

In [ ]:
df_train, df_val, df_test = split_dataset(df, 42)
mean = df_train["horsepower"].mean()
rmse0 = train_and_calculate_error_from_lesson(df_train, df_val, 0)
rmse_mean = train_and_calculate_error_from_lesson(df_train, df_val, mean)
print(f'RMSE with fill_value = 0: {rmse0}')
print(f'RMSE with fill_value = mean: {rmse_mean}')

- Now let's train a regularized linear regression.
- For this question, fill the NAs with 0.
- Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
- Use RMSE to evaluate the model on the validation dataset.
- Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.
- Which r gives the best RMSE?

In [ ]:
def train_lr_with_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [ ]:
def train_with_reg(df_train, df_val, r_to_try):
    df = cp_df_after_filling_hp(df_train, 0)
    X_train = df[features].to_numpy()
    y_train = df[target].to_numpy()
    
    df = cp_df_after_filling_hp(df_val, 0)
    X_val = df[features].to_numpy()
    y_val = df[target].to_numpy()
    
    for r in r_to_try:
        w0, w = train_lr_with_reg(X_train, y_train, r)
        
        y_pred = w0 + X_val.dot(w)
        score = rmse_from_lesson(y_val, y_pred)
        print(f"RMSE(r={r}) = {round(score, 4)}")

In [ ]:
r_to_try = [0, 0.01, 0.1, 1, 5, 10, 100]
train_with_reg(df_train, df_val, r_to_try)

- We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
- Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
- For each seed, do the train/validation/test split with 60%/20%/20% distribution.
- Fill the missing values with 0 and train a model without regularization.
- For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
- What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
- Round the result to 3 decimal digits (round(std, 3))

In [ ]:
rmse =[]
for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    df_train, df_val, df_test = split_dataset(df, seed)
    rmse.append(
        train_and_calculate_error_from_lesson(df_train, df_val, 0))
print(round(np.std(rmse),3))

- Split the dataset like previously, use seed 9.
- Combine train and validation datasets.
- Fill the missing values with 0 and train a model with r=0.001.
- What's the RMSE on the test dataset?

In [ ]:
seed = 9
r = 0.001
df_train, df_val, df_test = split_dataset(df, seed)

df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

train_with_reg(df_full_train, df_test, [r])